In [ ]:
# 必要なライブラリのインストール
!pip install sentence-transformers faiss-cpu scikit-learn pandas numpy japanize-matplotlib tabulate

In [ ]:
# RAG多言語・リランキング比較評価システム
# Google Colab環境用
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
import random
import json
import matplotlib.pyplot as plt
from collections import defaultdict
import seaborn as sns
from tabulate import tabulate

# Google Colab用の日本語フォント設定
try:
    import japanize_matplotlib
    japanize_matplotlib.japanize()
except:
    # フォント設定のフォールバック
    import matplotlib.font_manager as fm

    # 利用可能な日本語フォントを探す
    font_list = fm.findSystemFonts()
    japanese_fonts = []

    for font_path in font_list:
        try:
            font_prop = fm.FontProperties(fname=font_path)
            font_name = font_prop.get_name()
            if any(jp_char in font_name for jp_char in ['gothic', 'mincho', 'Noto', 'DejaVu']):
                japanese_fonts.append(font_name)
        except:
            continue

    # 日本語対応フォントを設定
    if japanese_fonts:
        plt.rcParams['font.family'] = japanese_fonts[0]
    else:
        plt.rcParams['font.family'] = 'DejaVu Sans'

    # 負の値も表示できるように設定
    plt.rcParams['axes.unicode_minus'] = False

sns.set_style("whitegrid")

class MultilingualQAGenerator:
    """多言語Q&Aデータセット生成クラス"""

    def __init__(self):
        # 日本語テンプレート
        self.ja_qa_templates = {
            "定義・説明": {
                "questions": [
                    "{}とは何ですか？",
                    "{}について説明してください",
                    "{}の定義を教えてください",
                    "{}とはどのようなものですか？"
                ],
                "answers": [
                    "{}とは、{topic_desc}のことです。{detail_desc}",
                    "{}について説明します。{topic_desc}であり、{detail_desc}",
                    "{}は{topic_desc}として定義されます。{detail_desc}"
                ]
            },
            "方法・手順": {
                "questions": [
                    "{}の方法を教えてください",
                    "{}のやり方は？",
                    "{}を行うにはどうすればいいですか？",
                    "{}の手順を説明してください"
                ],
                "answers": [
                    "{}の方法は以下の通りです。{method_desc}",
                    "{}を行うには、{method_desc}",
                    "{}の手順：{method_desc}"
                ]
            },
            "メリット・効果": {
                "questions": [
                    "{}のメリットは何ですか？",
                    "{}の効果を教えてください",
                    "{}の利点は？",
                    "{}にはどんな良い点がありますか？"
                ],
                "answers": [
                    "{}のメリットは{benefit_desc}です。",
                    "{}の効果として{benefit_desc}が挙げられます。",
                    "{}の利点は{benefit_desc}です。"
                ]
            },
            "注意点・デメリット": {
                "questions": [
                    "{}の注意点は何ですか？",
                    "{}で気をつけることは？",
                    "{}のデメリットは？",
                    "{}の問題点を教えてください"
                ],
                "answers": [
                    "{}の注意点は{caution_desc}です。",
                    "{}で気をつけるべきは{caution_desc}です。",
                    "{}のデメリットとして{caution_desc}があります。"
                ]
            },
            "比較": {
                "questions": [
                    "{}と{}の違いは何ですか？",
                    "{}と{}を比較してください",
                    "{}と{}はどう違いますか？"
                ],
                "answers": [
                    "{}と{}の違いは{compare_desc}です。",
                    "{}と{}を比較すると{compare_desc}となります。"
                ]
            }
        }

        # 英語テンプレート
        self.en_qa_templates = {
            "Definition": {
                "questions": [
                    "What is {}?",
                    "Please explain {}",
                    "Define {}",
                    "What does {} mean?"
                ],
                "answers": [
                    "{} is {topic_desc}. {detail_desc}",
                    "Let me explain {}. It is {topic_desc} and {detail_desc}",
                    "{} is defined as {topic_desc}. {detail_desc}"
                ]
            },
            "Method": {
                "questions": [
                    "How to {}?",
                    "What is the method for {}?",
                    "How do you {}?",
                    "Please explain the steps for {}"
                ],
                "answers": [
                    "The method for {} is as follows: {method_desc}",
                    "To {}, you should {method_desc}",
                    "The steps for {}: {method_desc}"
                ]
            },
            "Benefits": {
                "questions": [
                    "What are the benefits of {}?",
                    "What are the advantages of {}?",
                    "Why is {} good?",
                    "What are the positive aspects of {}?"
                ],
                "answers": [
                    "The benefits of {} are {benefit_desc}.",
                    "The advantages of {} include {benefit_desc}.",
                    "The positive aspects of {} are {benefit_desc}."
                ]
            },
            "Drawbacks": {
                "questions": [
                    "What are the disadvantages of {}?",
                    "What should I be careful about with {}?",
                    "What are the drawbacks of {}?",
                    "What are the potential problems with {}?"
                ],
                "answers": [
                    "The disadvantages of {} are {caution_desc}.",
                    "You should be careful about {caution_desc} with {}.",
                    "The drawbacks of {} include {caution_desc}."
                ]
            },
            "Comparison": {
                "questions": [
                    "What is the difference between {} and {}?",
                    "Compare {} and {}",
                    "How do {} and {} differ?"
                ],
                "answers": [
                    "The difference between {} and {} is {compare_desc}.",
                    "Comparing {} and {}, {compare_desc}."
                ]
            }
        }

        # 日本語トピック情報
        self.ja_topics = {
            "Python": {
                "topic_desc": "プログラミング言語の一つ",
                "detail_desc": "シンプルで読みやすい構文が特徴で、AI・データサイエンス・Web開発など幅広い分野で使用されています。",
                "method_desc": "まず基本構文を学び、実際にコードを書いて練習し、プロジェクトに取り組むことで習得できます。",
                "benefit_desc": "学習しやすく、豊富なライブラリがあり、需要が高いこと",
                "caution_desc": "実行速度が他の言語より遅い場合があること"
            },
            "機械学習": {
                "topic_desc": "コンピュータがデータから自動的にパターンを学習する技術",
                "detail_desc": "大量のデータを用いてアルゴリズムを訓練し、予測や分類を行うAI技術の一分野です。",
                "method_desc": "データの収集・前処理、モデルの選択・訓練、評価・改善のサイクルを繰り返します。",
                "benefit_desc": "自動化による効率化、複雑なパターンの発見、予測精度の向上",
                "caution_desc": "大量のデータが必要で、バイアスや過学習のリスクがあること"
            },
            "投資": {
                "topic_desc": "将来の利益を期待して資金を投入すること",
                "detail_desc": "株式、債券、不動産などの資産に資金を投入し、値上がりや配当による収益を狙う活動です。",
                "method_desc": "目標設定、リスク許容度の確認、分散投資、定期的な見直しを行います。",
                "benefit_desc": "資産増加の可能性、インフレ対策、複利効果",
                "caution_desc": "元本割れのリスクがあり、市場の変動に影響されること"
            },
            "ダイエット": {
                "topic_desc": "体重や体脂肪を減らすための取り組み",
                "detail_desc": "食事制限や運動を通じて健康的に体重を減らし、理想的な体型を目指す活動です。",
                "method_desc": "カロリー制限、バランスの良い食事、適度な運動、生活習慣の改善を組み合わせます。",
                "benefit_desc": "健康改善、自信向上、病気の予防効果",
                "caution_desc": "急激な減量は健康を害する可能性があり、リバウンドのリスクがあること"
            },
            "データベース": {
                "topic_desc": "データを効率的に格納・管理するシステム",
                "detail_desc": "大量の情報を構造化して保存し、必要な時に素早く検索・更新できるコンピュータシステムです。",
                "method_desc": "要件定義、設計、実装、テスト、運用の段階を経て構築します。",
                "benefit_desc": "データの一元管理、高速検索、データの整合性保証",
                "caution_desc": "設計ミスによる性能低下やセキュリティ脆弱性のリスクがあること"
            }
        }

        # 英語トピック情報
        self.en_topics = {
            "Python": {
                "topic_desc": "a programming language",
                "detail_desc": "It features simple and readable syntax and is used in a wide range of fields including AI, data science, and web development.",
                "method_desc": "learn basic syntax first, practice by writing actual code, and work on projects to master it",
                "benefit_desc": "easy to learn, rich libraries, and high demand",
                "caution_desc": "execution speed may be slower than other languages"
            },
            "Machine Learning": {
                "topic_desc": "technology where computers automatically learn patterns from data",
                "detail_desc": "It is a branch of AI technology that uses large amounts of data to train algorithms for prediction and classification.",
                "method_desc": "repeat the cycle of data collection and preprocessing, model selection and training, evaluation and improvement",
                "benefit_desc": "automation efficiency, discovery of complex patterns, improved prediction accuracy",
                "caution_desc": "requires large amounts of data and has risks of bias and overfitting"
            },
            "Investment": {
                "topic_desc": "putting money into something expecting future profits",
                "detail_desc": "It is an activity that aims to profit from capital gains and dividends by investing funds in assets such as stocks, bonds, and real estate.",
                "method_desc": "set goals, confirm risk tolerance, diversify investments, and review regularly",
                "benefit_desc": "potential for asset growth, inflation protection, compound effect",
                "caution_desc": "risk of principal loss and being affected by market volatility"
            },
            "Diet": {
                "topic_desc": "efforts to reduce weight and body fat",
                "detail_desc": "It is an activity that aims to achieve an ideal body shape by reducing weight healthily through dietary restrictions and exercise.",
                "method_desc": "combine calorie restriction, balanced diet, moderate exercise, and lifestyle improvements",
                "benefit_desc": "health improvement, confidence boost, disease prevention effects",
                "caution_desc": "rapid weight loss can harm health and has risk of rebound"
            },
            "Database": {
                "topic_desc": "a system for efficiently storing and managing data",
                "detail_desc": "It is a computer system that structures and stores large amounts of information, allowing quick search and updates when needed.",
                "method_desc": "go through the stages of requirements definition, design, implementation, testing, and operation",
                "benefit_desc": "centralized data management, fast search, data integrity assurance",
                "caution_desc": "risk of performance degradation and security vulnerabilities due to design mistakes"
            }
        }

        # 言語別の意図タイプマッピング
        self.intent_mapping = {
            "定義・説明": "Definition",
            "方法・手順": "Method",
            "メリット・効果": "Benefits",
            "注意点・デメリット": "Drawbacks",
            "比較": "Comparison"
        }

        # トピック名マッピング
        self.topic_mapping = {
            "Python": "Python",
            "機械学習": "Machine Learning",
            "投資": "Investment",
            "ダイエット": "Diet",
            "データベース": "Database"
        }

    def generate_qa_pairs(self, language='ja', num_pairs=1000):
        """指定された言語でQ&Aペアを生成"""
        qa_pairs = []
        qa_id = 1

        if language == 'ja':
            templates = self.ja_qa_templates
            topics_data = self.ja_topics
            topics = list(self.ja_topics.keys())
            intent_types = list(self.ja_qa_templates.keys())
        else:  # english
            templates = self.en_qa_templates
            topics_data = self.en_topics
            topics = list(self.en_topics.keys())
            intent_types = list(self.en_qa_templates.keys())

        # 各トピック・各意図タイプの組み合わせを生成
        pairs_per_combination = num_pairs // (len(topics) * len(intent_types))

        for topic in topics:
            topic_data = topics_data[topic]

            for intent_type in intent_types:
                template_data = templates[intent_type]

                if intent_type in ["比較", "Comparison"]:
                    # 比較の場合は特別処理
                    if language == 'ja':
                        other_topics = [t for t in topics if t != topic]
                        if other_topics:
                            other_topic = random.choice(other_topics)
                            question_template = random.choice(template_data["questions"])
                            answer_template = random.choice(template_data["answers"])

                            question = question_template.format(topic, other_topic)
                            answer = answer_template.format(
                                topic, other_topic,
                                compare_desc=f"{topic}は{topic_data['topic_desc']}で、{other_topic}とは異なる特徴を持ちます"
                            )
                    else:
                        other_topics = [t for t in topics if t != topic]
                        if other_topics:
                            other_topic = random.choice(other_topics)
                            question_template = random.choice(template_data["questions"])
                            answer_template = random.choice(template_data["answers"])

                            question = question_template.format(topic, other_topic)
                            answer = answer_template.format(
                                topic, other_topic,
                                compare_desc=f"{topic} is {topic_data['topic_desc']} and has different characteristics from {other_topic}"
                            )

                    qa_pairs.append({
                        "qa_id": qa_id,
                        "intent_type": intent_type,
                        "topic": topic,
                        "question": question,
                        "answer": answer,
                        "language": language
                    })
                    qa_id += 1
                else:
                    # 通常の場合
                    for _ in range(pairs_per_combination):
                        question_template = random.choice(template_data["questions"])
                        answer_template = random.choice(template_data["answers"])

                        if language == 'ja':
                            question = question_template.format(topic)
                            answer = answer_template.format(topic, **topic_data)
                        else:
                            # 英語の場合、動詞の活用を考慮
                            if "How to {}?" in question_template:
                                verb_topic = self._get_verb_form(topic)
                                question = question_template.format(verb_topic)
                            else:
                                question = question_template.format(topic)

                            if "To {}," in answer_template:
                                verb_topic = self._get_verb_form(topic)
                                answer = answer_template.format(verb_topic, **topic_data)
                            else:
                                answer = answer_template.format(topic, **topic_data)

                        qa_pairs.append({
                            "qa_id": qa_id,
                            "intent_type": intent_type,
                            "topic": topic,
                            "question": question,
                            "answer": answer,
                            "language": language
                        })
                        qa_id += 1

        # 残りのペアを追加
        non_comparison_intents = [i for i in intent_types if i not in ["比較", "Comparison"]]
        while len(qa_pairs) < num_pairs:
            topic = random.choice(topics)
            intent_type = random.choice(non_comparison_intents)
            topic_data = topics_data[topic]
            template_data = templates[intent_type]

            question_template = random.choice(template_data["questions"])
            answer_template = random.choice(template_data["answers"])

            if language == 'ja':
                question = question_template.format(topic)
                answer = answer_template.format(topic, **topic_data)
            else:
                if "How to {}?" in question_template:
                    verb_topic = self._get_verb_form(topic)
                    question = question_template.format(verb_topic)
                else:
                    question = question_template.format(topic)

                if "To {}," in answer_template:
                    verb_topic = self._get_verb_form(topic)
                    answer = answer_template.format(verb_topic, **topic_data)
                else:
                    answer = answer_template.format(topic, **topic_data)

            qa_pairs.append({
                "qa_id": qa_id,
                "intent_type": intent_type,
                "topic": topic,
                "question": question,
                "answer": answer,
                "language": language
            })
            qa_id += 1

        return qa_pairs[:num_pairs]

    def _get_verb_form(self, topic):
        """英語の動詞形を取得"""
        verb_forms = {
            "Python": "use Python",
            "Machine Learning": "do machine learning",
            "Investment": "invest",
            "Diet": "diet",
            "Database": "use databases"
        }
        return verb_forms.get(topic, f"use {topic}")

    def generate_test_queries(self, qa_pairs, language='ja', num_queries=100):
        """指定された言語でテストクエリを生成"""
        test_queries = []

        # クエリタイプ別の生成数
        query_types = {
            "exact_question": 25,
            "rephrase": 25,
            "keyword_search": 25,
            "intent_focused": 25
        }

        query_id = 1
        for query_type, count in query_types.items():
            for _ in range(count):
                qa_sample = random.choice(qa_pairs)

                query_text, expected_qa_id = self._generate_clear_query(
                    query_type, qa_sample, language
                )

                test_queries.append({
                    "query_id": query_id,
                    "query_type": query_type,
                    "query_text": query_text,
                    "expected_qa_id": expected_qa_id,
                    "expected_question": qa_sample["question"],
                    "expected_answer": qa_sample["answer"],
                    "topic": qa_sample["topic"],
                    "intent_type": qa_sample["intent_type"],
                    "language": language
                })
                query_id += 1

        return test_queries

    def _generate_clear_query(self, query_type, qa_sample, language):
        """言語に応じた明確なクエリを生成"""
        topic = qa_sample["topic"]
        intent_type = qa_sample["intent_type"]
        original_question = qa_sample["question"]
        qa_id = qa_sample["qa_id"]

        if query_type == "exact_question":
            return original_question, qa_id

        elif query_type == "rephrase":
            if language == 'ja':
                rephrase_patterns = {
                    "定義・説明": [f"{topic}って何？", f"{topic}とはどういう意味？"],
                    "方法・手順": [f"{topic}のコツは？", f"{topic}をするには？"],
                    "メリット・効果": [f"{topic}の良さは？", f"{topic}の価値は？"],
                    "注意点・デメリット": [f"{topic}の問題は？", f"{topic}で困ることは？"],
                    "比較": [f"{topic}と他の違いは？"]
                }
            else:
                rephrase_patterns = {
                    "Definition": [f"What's {topic}?", f"Meaning of {topic}?"],
                    "Method": [f"Tips for {topic}?", f"How to do {topic}?"],
                    "Benefits": [f"Why {topic}?", f"Value of {topic}?"],
                    "Drawbacks": [f"Problems with {topic}?", f"Issues with {topic}?"],
                    "Comparison": [f"Difference of {topic}?"]
                }

            patterns = rephrase_patterns.get(intent_type, [f"{topic}について" if language == 'ja' else f"About {topic}"])
            query_text = random.choice(patterns) if patterns else (f"{topic}について" if language == 'ja' else f"About {topic}")
            return query_text, qa_id

        elif query_type == "keyword_search":
            if language == 'ja':
                intent_keywords = {
                    "定義・説明": ["とは", "説明", "意味"],
                    "方法・手順": ["方法", "やり方", "手順"],
                    "メリット・効果": ["メリット", "効果", "利点"],
                    "注意点・デメリット": ["注意", "デメリット", "問題"],
                    "比較": ["違い", "比較"]
                }
            else:
                intent_keywords = {
                    "Definition": ["definition", "meaning", "what"],
                    "Method": ["method", "how", "steps"],
                    "Benefits": ["benefits", "advantages", "pros"],
                    "Drawbacks": ["disadvantages", "problems", "cons"],
                    "Comparison": ["difference", "comparison", "vs"]
                }

            keyword = random.choice(intent_keywords.get(intent_type, ["について" if language == 'ja' else "about"]))
            query_text = f"{topic} {keyword}"
            return query_text, qa_id

        elif query_type == "intent_focused":
            if language == 'ja':
                intent_queries = {
                    "定義・説明": f"{topic}の基本概念",
                    "方法・手順": f"{topic}の実践方法",
                    "メリット・効果": f"{topic}の利益",
                    "注意点・デメリット": f"{topic}のリスク",
                    "比較": f"{topic}の特徴"
                }
            else:
                intent_queries = {
                    "Definition": f"{topic} basics",
                    "Method": f"{topic} practice",
                    "Benefits": f"{topic} advantages",
                    "Drawbacks": f"{topic} risks",
                    "Comparison": f"{topic} features"
                }

            query_text = intent_queries.get(intent_type, f"{topic}について" if language == 'ja' else f"About {topic}")
            return query_text, qa_id

        return original_question, qa_id

class RerankingEvaluator:
    """リランキング機能付き評価クラス"""

    def __init__(self, ja_rag_system, en_rag_system):
        self.ja_rag_system = ja_rag_system
        self.en_rag_system = en_rag_system
        self.k = 3

        # リランキング用のCross-encoderモデル
        print("リランキング用Cross-encoderモデルを読み込み中...")
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        print("Cross-encoderモデル読み込み完了")

    def evaluate_with_reranking_comparison(self, ja_test_queries, en_test_queries):
        """リランキングありなしの比較評価"""
        print("リランキング比較評価を実行中...")

        results = {}

        # 日本語評価（リランキングなし・あり）
        print("Japan 日本語評価中...")
        ja_no_rerank = self._evaluate_single_language(ja_test_queries, self.ja_rag_system, "Japanese", use_reranking=False)
        ja_with_rerank = self._evaluate_single_language(ja_test_queries, self.ja_rag_system, "Japanese", use_reranking=True)

        # 英語評価（リランキングなし・あり）
        print("America 英語評価中...")
        en_no_rerank = self._evaluate_single_language(en_test_queries, self.en_rag_system, "English", use_reranking=False)
        en_with_rerank = self._evaluate_single_language(en_test_queries, self.en_rag_system, "English", use_reranking=True)

        results = {
            "japanese_no_rerank": ja_no_rerank,
            "japanese_with_rerank": ja_with_rerank,
            "english_no_rerank": en_no_rerank,
            "english_with_rerank": en_with_rerank
        }

        return results

    def _evaluate_single_language(self, test_queries, rag_system, language_name, use_reranking=False):
        """単一言語の評価（リランキングあり・なし対応）"""
        rerank_status = "with Reranking" if use_reranking else "without Reranking"
        print(f"  {language_name} {rerank_status} 評価中...")

        all_metrics = {
            'recall_at_3': [],
            'precision_at_3': [],
            'hit_at_3': [],
            'mrr': [],
            'exact_match': []
        }

        query_type_metrics = defaultdict(lambda: {
            'recall_at_3': [],
            'precision_at_3': [],
            'hit_at_3': [],
            'mrr': [],
            'exact_match': []
        })

        intent_type_metrics = defaultdict(lambda: {
            'recall_at_3': [],
            'precision_at_3': [],
            'hit_at_3': [],
            'mrr': [],
            'exact_match': []
        })

        detailed_results = []

        for i, query_data in enumerate(test_queries):
            if (i + 1) % 20 == 0:
                print(f"    {language_name} {rerank_status} 進捗: {i + 1}/{len(test_queries)} 完了")

            query_text = query_data["query_text"]
            expected_question = query_data["expected_question"]
            expected_answer = query_data["expected_answer"]
            query_type = query_data["query_type"]
            intent_type = query_data["intent_type"]

            # 検索実行（リランキングあり・なし）
            if use_reranking:
                search_results = self._search_with_reranking(rag_system, query_text, self.k)
            else:
                search_results = rag_system.search_semantic(query_text, self.k)

            # 期待されるQ&Aペアとマッチング
            metrics, found_rank = self._calculate_precise_metrics(
                search_results, expected_question, expected_answer
            )

            # 詳細結果を保存
            detailed_results.append({
                'query_id': query_data['query_id'],
                'query_text': query_text,
                'expected_question': expected_question,
                'found_rank': found_rank,
                'top3_questions': [r['question'] for r in search_results],
                'hit': metrics['hit_at_3'],
                'query_type': query_type,
                'intent_type': intent_type,
                'reranking': use_reranking
            })

            # 各メトリクスに追加
            for metric_name, value in metrics.items():
                all_metrics[metric_name].append(value)
                query_type_metrics[query_type][metric_name].append(value)
                intent_type_metrics[intent_type][metric_name].append(value)

        # 平均値計算
        results = {
            'overall': self._average_metrics(all_metrics),
            'by_query_type': {
                query_type: self._average_metrics(metrics)
                for query_type, metrics in query_type_metrics.items()
            },
            'by_intent_type': {
                intent_type: self._average_metrics(metrics)
                for intent_type, metrics in intent_type_metrics.items()
            },
            'detailed_results': detailed_results,
            'reranking': use_reranking
        }

        return results

    def _search_with_reranking(self, rag_system, query, final_k=3, initial_k=10):
        """リランキング付き検索"""
        # 1. 初期検索で多めに取得
        initial_results = rag_system.search_semantic(query, initial_k)

        # 2. Cross-encoderでリランキング
        query_doc_pairs = []
        for result in initial_results:
            # クエリと文書のペアを作成
            doc_text = f"{result['question']} {result['answer']}"
            query_doc_pairs.append([query, doc_text])

        # 3. Cross-encoderでスコア計算
        rerank_scores = self.reranker.predict(query_doc_pairs)

        # 4. スコアでソートして上位k件を取得
        scored_results = list(zip(initial_results, rerank_scores))
        scored_results.sort(key=lambda x: x[1], reverse=True)

        # 5. 最終結果を作成
        reranked_results = []
        for i, (result, score) in enumerate(scored_results[:final_k]):
            reranked_result = result.copy()
            reranked_result['rank'] = i + 1
            reranked_result['rerank_score'] = float(score)
            reranked_results.append(reranked_result)

        return reranked_results

    def _calculate_precise_metrics(self, search_results, expected_question, expected_answer):
        """期待される回答との精密マッチング"""
        metrics = {
            'recall_at_3': 0,
            'precision_at_3': 0,
            'hit_at_3': 0,
            'mrr': 0,
            'exact_match': 0
        }

        found_rank = None

        # 検索結果から期待されるQ&Aペアを探す
        for i, result in enumerate(search_results):
            # 質問と回答の両方が一致するかチェック
            if (result['question'] == expected_question and
                result['answer'] == expected_answer):
                found_rank = i + 1
                break
            # 質問のみ一致の場合も考慮
            elif result['question'] == expected_question:
                found_rank = i + 1
                break

        if found_rank is not None:
            metrics['recall_at_3'] = 1.0
            metrics['precision_at_3'] = 1.0 / self.k
            metrics['hit_at_3'] = 1.0
            metrics['mrr'] = 1.0 / found_rank

            if found_rank == 1:
                metrics['exact_match'] = 1.0

        return metrics, found_rank

    def _average_metrics(self, metrics_list):
        """メトリクスの平均値計算"""
        averaged = {}
        for metric_name, values in metrics_list.items():
            averaged[metric_name] = np.mean(values) if values else 0
        return averaged

    def print_reranking_comparison_results(self, results):
        """リランキング比較結果の表示"""
        print("\n" + "="*80)
        print("RAGシステム リランキング比較評価結果")
        print("="*80)

        # 4つのケースを取得
        ja_no_rerank = results["japanese_no_rerank"]
        ja_with_rerank = results["japanese_with_rerank"]
        en_no_rerank = results["english_no_rerank"]
        en_with_rerank = results["english_with_rerank"]

        # 全体Hit@3比較テーブル
        print("\n全体Hit@3パフォーマンス比較")
        print("-"*70)

        overall_comparison_data = [
            [
                "Japan 日本語",
                f"{ja_no_rerank['overall']['hit_at_3']:.3f}",
                f"{ja_with_rerank['overall']['hit_at_3']:.3f}",
                f"{(ja_with_rerank['overall']['hit_at_3'] - ja_no_rerank['overall']['hit_at_3']):+.3f}",
                "改善" if ja_with_rerank['overall']['hit_at_3'] > ja_no_rerank['overall']['hit_at_3'] else "変化なし"
            ],
            [
                "America 英語",
                f"{en_no_rerank['overall']['hit_at_3']:.3f}",
                f"{en_with_rerank['overall']['hit_at_3']:.3f}",
                f"{(en_with_rerank['overall']['hit_at_3'] - en_no_rerank['overall']['hit_at_3']):+.3f}",
                "改善" if en_with_rerank['overall']['hit_at_3'] > en_no_rerank['overall']['hit_at_3'] else "変化なし"
            ]
        ]

        print(tabulate(overall_comparison_data,
                      headers=["言語", "リランキングなし", "リランキングあり", "改善", "効果"],
                      tablefmt="fancy_grid", stralign="center"))

        # 詳細メトリクス比較
        print("\n詳細メトリクス比較")
        print("-"*70)

        metrics = ['recall_at_3', 'precision_at_3', 'hit_at_3', 'mrr', 'exact_match']
        metric_names = ['Recall@3', 'Precision@3', 'Hit@3', 'MRR', 'Exact Match']

        print("\nJapan 日本語詳細:")
        ja_detailed_data = []
        for metric, name in zip(metrics, metric_names):
            no_rerank = ja_no_rerank['overall'][metric]
            with_rerank = ja_with_rerank['overall'][metric]
            improvement = with_rerank - no_rerank
            ja_detailed_data.append([
                name,
                f"{no_rerank:.3f}",
                f"{with_rerank:.3f}",
                f"{improvement:+.3f}",
                "向上" if improvement > 0 else "低下" if improvement < 0 else "変化なし"
            ])

        print(tabulate(ja_detailed_data,
                      headers=["メトリクス", "リランキングなし", "リランキングあり", "改善", "結果"],
                      tablefmt="fancy_grid", stralign="center"))

        print("\nAmerica 英語詳細:")
        en_detailed_data = []
        for metric, name in zip(metrics, metric_names):
            no_rerank = en_no_rerank['overall'][metric]
            with_rerank = en_with_rerank['overall'][metric]
            improvement = with_rerank - no_rerank
            en_detailed_data.append([
                name,
                f"{no_rerank:.3f}",
                f"{with_rerank:.3f}",
                f"{improvement:+.3f}",
                "向上" if improvement > 0 else "低下" if improvement < 0 else "変化なし"
            ])

        print(tabulate(en_detailed_data,
                      headers=["メトリクス", "リランキングなし", "リランキングあり", "改善", "結果"],
                      tablefmt="fancy_grid", stralign="center"))

        # 意図タイプ別リランキング効果
        print("\n意図タイプ別リランキング効果")
        print("-"*70)

        # 日本語と英語の意図タイプをマッピング
        intent_mapping = {
            "定義・説明": "Definition",
            "方法・手順": "Method",
            "メリット・効果": "Benefits",
            "注意点・デメリット": "Drawbacks",
            "比較": "Comparison"
        }

        intent_rerank_data = []
        for ja_intent, en_intent in intent_mapping.items():
            # 日本語
            ja_no = ja_no_rerank['by_intent_type'].get(ja_intent, {}).get('hit_at_3', 0)
            ja_with = ja_with_rerank['by_intent_type'].get(ja_intent, {}).get('hit_at_3', 0)
            ja_improvement = ja_with - ja_no

            # 英語
            en_no = en_no_rerank['by_intent_type'].get(en_intent, {}).get('hit_at_3', 0)
            en_with = en_with_rerank['by_intent_type'].get(en_intent, {}).get('hit_at_3', 0)
            en_improvement = en_with - en_no

            intent_rerank_data.append([
                f"{ja_intent}",
                f"{ja_improvement:+.3f}",
                f"{en_improvement:+.3f}",
                "Japan" if ja_improvement > en_improvement else "America" if en_improvement > ja_improvement else "引き分け"
            ])

        print(tabulate(intent_rerank_data,
                      headers=["意図タイプ", "日本語改善", "英語改善", "より効果的"],
                      tablefmt="fancy_grid", stralign="center"))

        # 総合サマリー
        print(f"\nリランキング効果総合サマリー")
        print("-"*50)

        ja_overall_improvement = ja_with_rerank['overall']['hit_at_3'] - ja_no_rerank['overall']['hit_at_3']
        en_overall_improvement = en_with_rerank['overall']['hit_at_3'] - en_no_rerank['overall']['hit_at_3']

        print(f"Japan 日本語リランキング効果: {ja_overall_improvement:+.3f}")
        print(f"America 英語リランキング効果: {en_overall_improvement:+.3f}")

        if ja_overall_improvement > 0 and en_overall_improvement > 0:
            print("リランキングは両言語で効果あり！")
        elif ja_overall_improvement > 0 or en_overall_improvement > 0:
            better_lang = "日本語" if ja_overall_improvement > en_overall_improvement else "英語"
            print(f"リランキングは{better_lang}でより効果的")
        else:
            print("リランキングは両言語で効果なし")

        # 最高パフォーマンス
        best_score = max(
            ja_no_rerank['overall']['hit_at_3'],
            ja_with_rerank['overall']['hit_at_3'],
            en_no_rerank['overall']['hit_at_3'],
            en_with_rerank['overall']['hit_at_3']
        )

        if best_score == ja_with_rerank['overall']['hit_at_3']:
            best_config = "Japan 日本語 + リランキング"
        elif best_score == en_with_rerank['overall']['hit_at_3']:
            best_config = "America 英語 + リランキング"
        elif best_score == ja_no_rerank['overall']['hit_at_3']:
            best_config = "Japan 日本語（リランキングなし）"
        else:
            best_config = "America 英語（リランキングなし）"

        print(f"最高パフォーマンス: {best_config} ({best_score:.3f})")

    def create_reranking_comparison_visualization(self, results):
        """リランキング比較の可視化"""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('RAG Reranking Comparison: Japanese vs English, With vs Without Reranking',
                    fontsize=16, fontweight='bold')

        # データ準備
        ja_no_rerank = results["japanese_no_rerank"]
        ja_with_rerank = results["japanese_with_rerank"]
        en_no_rerank = results["english_no_rerank"]
        en_with_rerank = results["english_with_rerank"]

        # カラー設定
        ja_color = '#FF6B6B'  # 赤（日本語）
        en_color = '#4ECDC4'  # 青緑（英語）
        no_rerank_alpha = 0.6  # リランキングなしは薄く
        with_rerank_alpha = 1.0  # リランキングありは濃く

        # 1. 全体Hit@3比較
        ax = axes[0, 0]
        configs = ['JA\nNo Rerank', 'JA\nWith Rerank', 'EN\nNo Rerank', 'EN\nWith Rerank']
        hit3_values = [
            ja_no_rerank['overall']['hit_at_3'],
            ja_with_rerank['overall']['hit_at_3'],
            en_no_rerank['overall']['hit_at_3'],
            en_with_rerank['overall']['hit_at_3']
        ]
        colors = [ja_color, ja_color, en_color, en_color]

        # バーを個別に描画してアルファ値を設定
        bars = []
        alphas = [no_rerank_alpha, with_rerank_alpha, no_rerank_alpha, with_rerank_alpha]
        for i, (config, value, color, alpha) in enumerate(zip(configs, hit3_values, colors, alphas)):
            bar = ax.bar(i, value, color=color, alpha=alpha)
            bars.extend(bar)

        ax.set_title('Overall Hit@3 Comparison', fontsize=14, fontweight='bold')
        ax.set_ylabel('Hit@3 Score')
        ax.set_xticks(range(len(configs)))
        ax.set_xticklabels(configs)
        ax.set_ylim(0, 1)

        # 値をバーの上に表示
        for i, value in enumerate(hit3_values):
            ax.text(i, value + 0.02, f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

        # 2. リランキング効果（改善度）
        ax = axes[0, 1]
        languages = ['Japanese', 'English']
        ja_improvement = ja_with_rerank['overall']['hit_at_3'] - ja_no_rerank['overall']['hit_at_3']
        en_improvement = en_with_rerank['overall']['hit_at_3'] - en_no_rerank['overall']['hit_at_3']
        improvements = [ja_improvement, en_improvement]
        colors_imp = [ja_color, en_color]

        bars = ax.bar(languages, improvements, color=colors_imp)
        ax.set_title('Reranking Improvement Effect', fontsize=14, fontweight='bold')
        ax.set_ylabel('Hit@3 Improvement')
        ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

        # 改善値をバーに表示
        for bar, improvement in zip(bars, improvements):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, height + (0.01 if height > 0 else -0.02),
                   f'{improvement:+.3f}', ha='center', va='bottom' if height > 0 else 'top',
                   fontweight='bold')

        # 3. 詳細メトリクス比較（日本語）
        ax = axes[1, 0]
        metrics = ['recall_at_3', 'precision_at_3', 'hit_at_3', 'mrr', 'exact_match']
        metric_names = ['Recall', 'Precision', 'Hit@3', 'MRR', 'EM']

        ja_no_values = [ja_no_rerank['overall'][metric] for metric in metrics]
        ja_with_values = [ja_with_rerank['overall'][metric] for metric in metrics]

        x = np.arange(len(metric_names))
        width = 0.35

        bars1 = ax.bar(x - width/2, ja_no_values, width, label='No Reranking',
                      color=ja_color, alpha=no_rerank_alpha)
        bars2 = ax.bar(x + width/2, ja_with_values, width, label='With Reranking',
                      color=ja_color, alpha=with_rerank_alpha)

        ax.set_title('Japanese: Detailed Metrics Comparison', fontsize=14, fontweight='bold')
        ax.set_ylabel('Score')
        ax.set_xlabel('Metrics')
        ax.set_xticks(x)
        ax.set_xticklabels(metric_names)
        ax.legend()
        ax.set_ylim(0, 1)

        # 4. 詳細メトリクス比較（英語）
        ax = axes[1, 1]

        en_no_values = [en_no_rerank['overall'][metric] for metric in metrics]
        en_with_values = [en_with_rerank['overall'][metric] for metric in metrics]

        bars1 = ax.bar(x - width/2, en_no_values, width, label='No Reranking',
                      color=en_color, alpha=no_rerank_alpha)
        bars2 = ax.bar(x + width/2, en_with_values, width, label='With Reranking',
                      color=en_color, alpha=with_rerank_alpha)

        ax.set_title('English: Detailed Metrics Comparison', fontsize=14, fontweight='bold')
        ax.set_ylabel('Score')
        ax.set_xlabel('Metrics')
        ax.set_xticks(x)
        ax.set_xticklabels(metric_names)
        ax.legend()
        ax.set_ylim(0, 1)

        plt.tight_layout()
        plt.show()

class MultilingualRAGSystemWithReranking:
    """リランキング機能付き多言語RAGシステム"""

    def __init__(self, model_name='paraphrase-multilingual-MiniLM-L12-v2'):
        self.model = SentenceTransformer(model_name)
        self.df = None
        self.embeddings = None
        self.index = None

    def load_data_from_dict(self, qa_data_list, language='ja'):
        """辞書リストからデータを読み込み"""
        simplified_data = []
        for qa in qa_data_list:
            simplified_data.append({
                'question': qa['question'],
                'answer': qa['answer']
            })

        self.df = pd.DataFrame(simplified_data)
        lang_name = "日本語" if language == 'ja' else "English"
        print(f"{lang_name}データを読み込みました: {len(self.df)}件")
        return self.df

    def create_embeddings(self, language='ja'):
        """埋め込み作成"""
        documents = []
        for _, row in self.df.iterrows():
            if language == 'ja':
                doc = f"質問: {row['question']} 回答: {row['answer']}"
            else:
                doc = f"Question: {row['question']} Answer: {row['answer']}"
            documents.append(doc)

        lang_name = "日本語" if language == 'ja' else "English"
        print(f"{lang_name}埋め込みを作成中...")
        self.embeddings = self.model.encode(documents, show_progress_bar=True)

        # FAISSインデックス作成
        dimension = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)

        # 正規化
        normalized_embeddings = self.embeddings / np.linalg.norm(self.embeddings, axis=1, keepdims=True)
        self.index.add(normalized_embeddings.astype('float32'))

        print(f"{lang_name}FAISSインデックス作成完了")

    def search_semantic(self, query, top_k=3):
        """セマンティック検索"""
        query_embedding = self.model.encode([query])
        query_embedding = query_embedding / np.linalg.norm(query_embedding, axis=1, keepdims=True)

        scores, indices = self.index.search(query_embedding.astype('float32'), top_k)

        results = []
        for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
            result = {
                'rank': i + 1,
                'score': float(score),
                'question': self.df.iloc[idx]['question'],
                'answer': self.df.iloc[idx]['answer']
            }
            results.append(result)

        return results

def main():
    """メイン実行関数"""
    print("RAG多言語・リランキング比較評価システムを開始します")
    print("="*70)

    # 1. データ生成
    generator = MultilingualQAGenerator()

    # 日本語データ生成
    ja_qa_pairs = generator.generate_qa_pairs(language='ja', num_pairs=1000)
    ja_simple = [{'question': qa['question'], 'answer': qa['answer']} for qa in ja_qa_pairs]
    ja_df = pd.DataFrame(ja_simple)
    ja_df.to_csv('reranking_ja_qa_1000.csv', index=False, encoding='utf-8')

    # 英語データ生成
    en_qa_pairs = generator.generate_qa_pairs(language='en', num_pairs=1000)
    en_simple = [{'question': qa['question'], 'answer': qa['answer']} for qa in en_qa_pairs]
    en_df = pd.DataFrame(en_simple)
    en_df.to_csv('reranking_en_qa_1000.csv', index=False, encoding='utf-8')

    # 日本語テストクエリ
    ja_test_queries = generator.generate_test_queries(ja_qa_pairs, language='ja', num_queries=100)
    with open('reranking_ja_test_queries_100.json', 'w', encoding='utf-8') as f:
        json.dump(ja_test_queries, f, ensure_ascii=False, indent=2)

    # 英語テストクエリ
    en_test_queries = generator.generate_test_queries(en_qa_pairs, language='en', num_queries=100)
    with open('reranking_en_test_queries_100.json', 'w', encoding='utf-8') as f:
        json.dump(en_test_queries, f, ensure_ascii=False, indent=2)

    # 日本語RAGシステム
    ja_rag_system = MultilingualRAGSystemWithReranking()
    ja_rag_system.load_data_from_dict(ja_qa_pairs, language='ja')
    ja_rag_system.create_embeddings(language='ja')

    # 英語RAGシステム
    en_rag_system = MultilingualRAGSystemWithReranking()
    en_rag_system.load_data_from_dict(en_qa_pairs, language='en')
    en_rag_system.create_embeddings(language='en')

    # 2. リランキング比較評価実行
    evaluator = RerankingEvaluator(ja_rag_system, en_rag_system)
    reranking_results = evaluator.evaluate_with_reranking_comparison(ja_test_queries, en_test_queries)

    # 3. 結果表示
    evaluator.print_reranking_comparison_results(reranking_results)

    # 4. 可視化
    evaluator.create_reranking_comparison_visualization(reranking_results)

    # 5. 結果保存
    save_results = {}
    for config_name, config_results in reranking_results.items():
        save_results[config_name] = {
            'overall': config_results['overall'],
            'by_query_type': config_results['by_query_type'],
            'by_intent_type': config_results['by_intent_type'],
            'reranking': config_results['reranking']
        }

    with open('reranking_comparison_results.json', 'w', encoding='utf-8') as f:
        json.dump(save_results, f, ensure_ascii=False, indent=2)

    # 統計とサマリー表示
    print("\n最終結果サマリー")
    print("="*50)

    configs = [
        ("Japan 日本語（リランキングなし）", reranking_results["japanese_no_rerank"]),
        ("Japan 日本語（リランキングあり）", reranking_results["japanese_with_rerank"]),
        ("America 英語（リランキングなし）", reranking_results["english_no_rerank"]),
        ("America 英語（リランキングあり）", reranking_results["english_with_rerank"])
    ]

    best_config = None
    best_score = 0

    for config_name, config_result in configs:
        score = config_result['overall']['hit_at_3']
        print(f"{config_name}: {score:.3f}")
        if score > best_score:
            best_score = score
            best_config = config_name

    print(f"\n最高パフォーマンス: {best_config} ({best_score:.3f})")

    # リランキング効果分析
    ja_improvement = (reranking_results["japanese_with_rerank"]['overall']['hit_at_3'] -
                     reranking_results["japanese_no_rerank"]['overall']['hit_at_3'])
    en_improvement = (reranking_results["english_with_rerank"]['overall']['hit_at_3'] -
                     reranking_results["english_no_rerank"]['overall']['hit_at_3'])

    print(f"\nリランキング効果:")
    print(f"  Japan 日本語: {ja_improvement:+.3f}")
    print(f"  America 英語: {en_improvement:+.3f}")

    return ja_rag_system, en_rag_system, reranking_results

if __name__ == "__main__":
    ja_system, en_system, results = main()